In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, re, json, pandas as pd, logging
import os
from urllib.parse import quote, unquote
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver():
    """Configure le driver Chrome"""
    options = Options()
    
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        return None

# --------------------------
# FONCTIONS DE SCRAPING GOOGLE AMÉLIORÉES
# --------------------------
def search_google_videos_2025(driver, query, max_results=100):
    """Recherche des vidéos sur Google pour septembre 2025 - VERSION AMÉLIORÉE"""
    try:
        # Encoder la requête pour l'URL
        encoded_query = quote(query)
        url = f"https://www.google.com/search?q={encoded_query}&tbm=vid&num=100"
        
        logging.info(f"🔍 Recherche Google 2025: {query}")
        driver.get(url)
        time.sleep(4)
        
        # Accepter les cookies si nécessaire
        try:
            accept_button = driver.find_element(By.XPATH, "//button[contains(., 'Tout accepter') or contains(., 'Accept all') or contains(., 'Accepter tout')]")
            accept_button.click()
            time.sleep(2)
        except:
            pass
        
        # Scroll agressif pour charger plus de résultats
        logging.info("📜 Scroll agressif pour charger plus de résultats...")
        for i in range(10):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)
        
        # Essayer de cliquer sur "Plus de résultats" si disponible
        try:
            more_results = driver.find_elements(By.XPATH, "//a[contains(., 'Plus de résultats') or contains(., 'More results')]")
            if more_results:
                more_results[0].click()
                time.sleep(3)
                # Rescroll après avoir cliqué
                for i in range(5):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(1)
        except:
            pass
        
        return extract_video_links_improved(driver, max_results)
        
    except Exception as e:
        logging.error(f"❌ Erreur recherche Google 2025: {e}")
        return []

def extract_video_links_improved(driver, max_results):
    """Extrait les liens vidéo des résultats Google - VERSION AMÉLIORÉE"""
    video_links = []
    
    try:
        # Méthode 1: Chercher dans tous les liens de la page
        all_links = driver.find_elements(By.TAG_NAME, "a")
        
        for link in all_links:
            if len(video_links) >= max_results:
                break
                
            try:
                href = link.get_attribute("href")
                if href and ("x.com" in href or "twitter.com" in href):
                    # Nettoyer l'URL
                    clean_url = clean_twitter_url(href)
                    if clean_url and clean_url not in video_links:
                        video_links.append(clean_url)
                        logging.info(f"✅ Lien trouvé: {clean_url}")
            except:
                continue
        
        # Méthode 2: Chercher spécifiquement dans les résultats vidéo
        if len(video_links) < max_results:
            try:
                video_elements = driver.find_elements(By.CSS_SELECTOR, "div.g, div.video-result, div[data-ved], div[role='heading']")
                
                for element in video_elements:
                    if len(video_links) >= max_results:
                        break
                        
                    try:
                        link_selectors = ["a[href]", "a"]
                        for selector in link_selectors:
                            try:
                                links = element.find_elements(By.CSS_SELECTOR, selector)
                                for link in links:
                                    href = link.get_attribute("href")
                                    if href and ("x.com" in href or "twitter.com" in href):
                                        clean_url = clean_twitter_url(href)
                                        if clean_url and clean_url not in video_links:
                                            video_links.append(clean_url)
                                            logging.info(f"✅ Lien trouvé (méthode 2): {clean_url}")
                                            break
                            except:
                                continue
                    except:
                        continue
            except:
                pass
        
        # Méthode 3: Chercher par texte dans les liens
        if len(video_links) < max_results:
            try:
                twitter_links = driver.find_elements(By.XPATH, "//a[contains(@href, 'x.com') or contains(@href, 'twitter.com')]")
                for link in twitter_links:
                    if len(video_links) >= max_results:
                        break
                    href = link.get_attribute("href")
                    clean_url = clean_twitter_url(href)
                    if clean_url and clean_url not in video_links:
                        video_links.append(clean_url)
                        logging.info(f"✅ Lien trouvé (méthode 3): {clean_url}")
            except:
                pass
                
    except Exception as e:
        logging.error(f"❌ Erreur extraction liens vidéo: {e}")
    
    logging.info(f"📊 Total liens extraits: {len(video_links)}")
    return video_links

def clean_twitter_url(url):
    """Nettoie et valide les URLs Twitter"""
    try:
        # Si c'est une URL Google redirect, extraire le vrai URL
        if "google.com/url" in url:
            match = re.search(r'url=([^&]+)', url)
            if match:
                url = unquote(match.group(1))
        
        # Garder seulement les URLs Twitter/X
        if "x.com/" in url or "twitter.com/" in url:
            # S'assurer que c'est un lien de statut
            if "/status/" in url:
                # Prendre seulement la partie avant les paramètres
                clean_url = url.split('?')[0]
                return clean_url
                
    except Exception as e:
        logging.debug(f"Erreur nettoyage URL: {e}")
    
    return None

def extract_tweet_data_quick(driver):
    """Extrait rapidement les données du tweet (optimisé pour la vitesse)"""
    tweet_data = {
        "title": "",
        "timestamp": "",
        "author": "Hespress",
        "metrics": {"likes": 0, "retweets": 0}
    }
    
    try:
        # Titre du tweet - méthode rapide
        try:
            title_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweetText"], article div[dir="auto"]')
            if title_elements:
                tweet_data["title"] = title_elements[0].text[:500]  # Limiter la longueur
        except:
            pass
        
        # Date - méthode rapide
        try:
            time_elements = driver.find_elements(By.TAG_NAME, "time")
            if time_elements:
                tweet_data["timestamp"] = time_elements[0].get_attribute("datetime")
        except:
            pass
            
    except Exception as e:
        logging.debug(f"Erreur extraction rapide données tweet: {e}")
    
    return tweet_data

# --------------------------
# SCRAPER PRINCIPAL OPTIMISÉ
# --------------------------
def scrape_hespress_videos_september_2025_max():
    """Scrape le MAXIMUM de vidéos Hespress de septembre 2025"""
    
    driver = setup_driver()
    if not driver:
        logging.error("❌ Impossible de créer le driver Chrome")
        return
    
    try:
        # REQUÊTES ÉTENDUES POUR MAXIMISER LES RÉSULTATS
        queries = [
            "hespress video twitter september 2025",
            "hespress x.com video septembre 2025", 
            "hespress twitter video month september 2025",
            "site:x.com hespress video september 2025",
            "hespress vidéo twitter septembre 2025",
            '"hespress" "video" "september 2025" site:x.com',
            'hespress "septembre 2025" video twitter',
            'hespress "2025-09" video twitter',
            'hespress video "sept 2025" twitter',
            'hespress "september 2025" x.com video',
            # Nouvelles requêtes étendues
            'hespress.com video twitter september 2025',
            'hespress maroc video twitter 2025',
            'hespress actualités video septembre 2025',
            'hespress news video twitter 2025',
            '"hespress" "septembre" "2025" "video"',
            'hespress clip video twitter 2025',
            'hespress reportage video septembre'
        ]
        
        all_video_links = []
        
        for i, query in enumerate(queries, 1):
            logging.info(f"🎯 Recherche {i}/{len(queries)}: {query}")
            links = search_google_videos_2025(driver, query, max_results=150)  # Augmenté à 150
            all_video_links.extend(links)
            
            # Sauvegarde intermédiaire des URLs
            with open("hespress_urls_temporaire.txt", "w", encoding="utf-8") as f:
                for url in list(set(all_video_links)):
                    f.write(url + "\n")
            
            logging.info(f"📊 Progression: {len(set(all_video_links))} liens uniques accumulés")
            time.sleep(2)  # Pause courte entre les recherches
        
        # Supprimer les doublons
        unique_links = list(set(all_video_links))
        logging.info(f"🎉 TOTAL LIENS UNIQUES TROUVÉS: {len(unique_links)}")
        
        if not unique_links:
            logging.warning("❌ Aucun lien trouvé pour septembre 2025.")
            return
        
        # Sauvegarder la liste complète des URLs
        with open("hespress_video_urls_septembre_2025_COMPLET.txt", "w", encoding="utf-8") as f:
            for url in unique_links:
                f.write(url + "\n")
        
        # FILTRAGE SEPTEMBRE 2025 - Version optimisée
        september_2025_results = []
        total_a_analyser = len(unique_links)
        
        logging.info(f"🔍 Début de l'analyse et filtrage septembre 2025 sur {total_a_analyser} liens...")
        
        for i, link in enumerate(unique_links, 1):
            try:
                if i % 10 == 0:
                    logging.info(f"📊 Analyse {i}/{total_a_analyser} - {len(september_2025_results)} vidéos septembre 2025 trouvées")
                
                driver.get(link)
                time.sleep(2)  # Réduit le temps d'attente
                
                # Extraction RAPIDE des données
                tweet_data = extract_tweet_data_quick(driver)
                
                # FILTRE SEPTEMBRE 2025
                timestamp = tweet_data.get("timestamp", "")
                if "2025-09" in timestamp:
                    video_data = {
                        "tweet_url": link,
                        "title": tweet_data["title"],
                        "timestamp": timestamp,
                        "author": tweet_data["author"],
                        "metrics": tweet_data["metrics"],
                        "success": True,
                        "scraped_at": datetime.now().isoformat()
                    }
                    september_2025_results.append(video_data)
                    logging.info(f"✅ SEPTEMBRE 2025: {timestamp} - {tweet_data['title'][:100]}...")
                
                # Sauvegarde incrémentale
                if i % 20 == 0:
                    with open("hespress_septembre_2025_temp.json", "w", encoding="utf-8") as f:
                        json.dump(september_2025_results, f, ensure_ascii=False, indent=2)
                
            except Exception as e:
                logging.debug(f"❌ Erreur sur {link}: {e}")
                continue
        
        # SAUVEGARDE FINALE
        save_results_max(september_2025_results, len(unique_links))
        
        # STATISTIQUES DÉTAILLÉES
        logging.info(f"📊 RAPPORT FINAL SEPTEMBRE 2025:")
        logging.info(f"   🔗 Total liens analysés: {len(unique_links)}")
        logging.info(f"   ✅ Vidéos septembre 2025 validées: {len(september_2025_results)}")
        logging.info(f"   📈 Taux de réussite: {len(september_2025_results)/len(unique_links)*100:.1f}%")
        
        if september_2025_results:
            dates = [result["timestamp"] for result in september_2025_results if result.get("timestamp")]
            if dates:
                logging.info(f"   📅 Plage de dates: {min(dates)} à {max(dates)}")
        
    except Exception as e:
        logging.error(f"❌ Erreur générale: {e}")
    finally:
        driver.quit()
        logging.info("🔒 Navigateur fermé")

def save_results_max(results, total_analyses):
    """Sauvegarde les résultats avec statistiques détaillées"""
    try:
        # JSON complet avec métadonnées
        output_data = {
            "metadata": {
                "scraping_date": datetime.now().isoformat(),
                "total_urls_analyzed": total_analyses,
                "september_2025_videos_found": len(results),
                "success_rate": f"{(len(results)/total_analyses*100):.1f}%" if total_analyses > 0 else "0%"
            },
            "videos": results
        }
        
        with open("hespress_videos_septembre_2025_FINAL.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
        
        # CSV détaillé
        if results:
            csv_data = []
            for result in results:
                csv_data.append({
                    "tweet_url": result.get("tweet_url", ""),
                    "title": result.get("title", ""),
                    "author": result.get("author", ""),
                    "timestamp": result.get("timestamp", ""),
                    "likes": result.get("metrics", {}).get("likes", 0),
                    "retweets": result.get("metrics", {}).get("retweets", 0),
                    "scraped_at": result.get("scraped_at", "")
                })
            
            df = pd.DataFrame(csv_data)
            df.to_csv("hespress_videos_septembre_2025_FINAL.csv", index=False, encoding="utf-8-sig")
        
        # Fichier texte simple avec URLs
        with open("hespress_urls_septembre_2025_FINAL.txt", "w", encoding="utf-8") as f:
            for result in results:
                f.write(result.get("tweet_url", "") + "\n")
        
        logging.info("💾 FICHIERS FINAUX SAUVEGARDÉS:")
        logging.info("   - hespress_videos_septembre_2025_FINAL.json")
        logging.info("   - hespress_videos_septembre_2025_FINAL.csv")
        logging.info("   - hespress_urls_septembre_2025_FINAL.txt")
        logging.info("   - hespress_video_urls_septembre_2025_COMPLET.txt (tous les liens)")
        
    except Exception as e:
        logging.error(f"❌ Erreur sauvegarde: {e}")

# --------------------------
# VÉRIFICATION RAPIDE
# --------------------------
def check_results_quick():
    """Vérification rapide des résultats"""
    try:
        files = {
            "Fichier JSON complet": "hespress_videos_septembre_2025_FINAL.json",
            "Fichier CSV": "hespress_videos_septembre_2025_FINAL.csv", 
            "URLs septembre 2025": "hespress_urls_septembre_2025_FINAL.txt",
            "Tous les liens trouvés": "hespress_video_urls_septembre_2025_COMPLET.txt"
        }
        
        for nom, fichier in files.items():
            if os.path.exists(fichier):
                taille = os.path.getsize(fichier)
                print(f"✅ {nom}: {fichier} ({taille} octets)")
                
                if fichier.endswith('.json') and taille > 0:
                    with open(fichier, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if "videos" in data:
                            print(f"   📊 {len(data['videos'])} vidéos septembre 2025")
                        else:
                            print(f"   📊 {len(data)} vidéos septembre 2025")
                elif fichier.endswith('.txt'):
                    with open(fichier, 'r', encoding='utf-8') as f:
                        lignes = f.readlines()
                        print(f"   🔗 {len(lignes)} URLs")
            else:
                print(f"❌ {nom}: {fichier} - NON TROUVÉ")
                
    except Exception as e:
        print(f"❌ Erreur vérification: {e}")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    print("=" * 70)
    print("🎥 SCRAPER MAXIMAL HESPRESS SEPTEMBRE 2025")
    print("=" * 70)
    print("⚠️  Ce script va:")
    print("   - Utiliser 16 requêtes Google différentes")
    print("   - Scroller agressivement pour max de résultats") 
    print("   - Analyser jusqu'à 150 liens par requête")
    print("   - Filtrer AUTOMATIQUEMENT septembre 2025")
    print("   - Sauvegarder tous les liens trouvés")
    print("=" * 70)
    
    print("Options:")
    print("1. 🚀 Lancer le scraping MAXIMAL (recommandé)")
    print("2. 🔍 Vérifier les résultats existants")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        print("🚀 LANCEMENT DU SCRAPING MAXIMAL...")
        print("⏰ Cette opération peut prendre 15-30 minutes.")
        print("📈 Objectif: MAXIMISER le nombre de liens septembre 2025")
        confirmation = input("Confirmez-vous? (o/n): ").strip().lower()
        
        if confirmation == 'o':
            scrape_hespress_videos_september_2025_max()
        else:
            print("❌ Opération annulée.")
    elif choix == "2":
        print("🔍 Vérification des fichiers...")
        check_results_quick()
    else:
        print("❌ Choix invalide. Veuillez choisir 1 ou 2.")

🎥 SCRAPER MAXIMAL HESPRESS SEPTEMBRE 2025
⚠️  Ce script va:
   - Utiliser 16 requêtes Google différentes
   - Scroller agressivement pour max de résultats
   - Analyser jusqu'à 150 liens par requête
   - Filtrer AUTOMATIQUEMENT septembre 2025
   - Sauvegarder tous les liens trouvés
Options:
1. 🚀 Lancer le scraping MAXIMAL (recommandé)
2. 🔍 Vérifier les résultats existants



Votre choix (1 ou 2):  1


🚀 LANCEMENT DU SCRAPING MAXIMAL...
⏰ Cette opération peut prendre 15-30 minutes.
📈 Objectif: MAXIMISER le nombre de liens septembre 2025


Confirmez-vous? (o/n):  o


2025-10-25 19:10:08,141 - INFO - ====== WebDriver manager ======
2025-10-25 19:10:13,098 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-25 19:10:13,253 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-25 19:10:13,402 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32/chromedriver.exe] found in cache
2025-10-25 19:10:14,639 - INFO - 🎯 Recherche 1/17: hespress video twitter september 2025
2025-10-25 19:10:14,641 - INFO - 🔍 Recherche Google 2025: hespress video twitter september 2025
2025-10-25 19:10:20,289 - INFO - 📜 Scroll agressif pour charger plus de résultats...
2025-10-25 19:10:47,883 - INFO - ✅ Lien trouvé: https://x.com/hespress/status/1973155430911357254
2025-10-25 19:11:38,916 - INFO - ✅ Lien trouvé (méthode 3): https://x.com/hespress/status/1973170561812668582
2025-10-25 19:11:38,980 - INFO - ✅ Lien trouvé (méthode 3): https://x.com/hespress/status/1966849350472266010
2025-10-25 19:11:39,025 - I